# 실습 4: FourCastNet을 활용한 AI 기상 예측 · 한국어판

사전 학습된 FourCastNet으로 48시간 기상 예측을 생성하고 ERA5 재분석과 비교하며, 예측 선행 시간에 따라 오차가 어떻게 변하는지 살펴봅니다.

**2022년 9월 1일 00:00 UTC**의 대기 상태에서 시작합니다. 모델은 미래 ERA5 장을 보지 않고 6시간 간격으로 여덟 번 예측합니다.

이 실습을 마치면 다음을 할 수 있습니다.

- 신경 연산자가 전 지구 기상 상태를 다음 상태로 매핑하는 방법을 설명합니다.
- 예측을 재분석 및 단순한 지속성 기준 예측과 비교합니다.
- 보기 좋은 애니메이션만으로는 드러나지 않을 수 있는 예측 오차를 찾아냅니다.

이 실습은 학습이 아니라 **추론과 검증**을 다룹니다. ERA5는 관측과 수치 모델을 결합한 재분석 자료이며 직접 관측 자료가 아닙니다.

[시작 안내](../../Start_Here.ipynb) · 이전: [실습 3: 열전도](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · 다음: [도전 과제 1: 파동 역학](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)

## 현재 기상 상태에서 다음 상태로

FourCastNet은 **적응형 푸리에 신경 연산자(AFNO)**를 사용합니다. 각 층은 푸리에 공간에서 공간 정보를 혼합합니다. 여기서는 전 지구 **720 × 1440** 격자의 지표면 및 기압면 기상장을 입력받는 NVIDIA의 사전 학습된 **26채널 체크포인트**를 사용합니다.

![전 지구 기상장 26개의 격자와 6시간 간격의 예측 여덟 단계. 동아시아는 검증을 위해 잘라낸 영역이며 모델의 전체 영역이 아닙니다.](../../ETC/assets/teaching/lab4-domain.svg)

**코드에서 격자 찾기:** [run_forecast.py](source_code/run_forecast.py)의 `LATITUDE`, `LONGITUDE`, `load_initial()`을 살펴보세요. 동아시아 영역은 [evaluate_weather.py](source_code/evaluate_weather.py)의 `EAST_ASIA`와 `region_indices()`에 정의되어 있습니다.

### 코드 따라가기: 추론 후 검증

![초기 ERA5 자료에서 고정된 PhysicsNeMo AFNO 모델, 반복 예측, 별도의 ERA5 검증으로 이어지는 코드 흐름.](../../ETC/assets/teaching/lab4-workflow.svg)

**코드에서 따라가기:** [run_forecast.py](source_code/run_forecast.py)의 `load_model()` → `main()` → `torch.inference_mode()`, 이어서 [evaluate_weather.py](source_code/evaluate_weather.py)의 `load_pair()` → `evaluate()`를 살펴보세요. 노트북의 `run_script()`가 이 프로그램들을 실행합니다.

출력은 변화량이 아니라 다음 상태 전체입니다. 각 예측이 다음 입력이 되므로 오차가 전파될 수 있습니다. 미래 ERA5 장은 검증에만 사용합니다. 실습 1–3과 달리 이 실습은 모델 가중치를 갱신하거나 PINN 손실을 사용하지 않습니다.

## 1. 모델과 데이터 준비

**AI4Sci PhysicsNeMo** 커널을 사용하세요. CUDA를 사용할 수 있으면 기본 장치로 선택합니다. 환경 설정 셀에 선택한 장치가 표시됩니다. CPU에서도 실행할 수 있지만 더 느립니다.

처음 준비할 때 체크포인트(약 301 MB)와 이 사례의 ERA5 원본 데이터(압축 크기 약 622 MB)를 내려받습니다. 캐시된 파일은 재사용합니다. 이 파일들이 준비된 Launchable에서는 다시 내려받을 필요가 없습니다. 다운로드 시간은 예측 시간과 별도입니다.

이 노트북은 제공된 강의 환경을 사용하며 패키지를 설치하지 않습니다.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import uuid

import numpy as np
import torch
from IPython.display import Image, Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'Start_Here.ipynb').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the bootcamp repository.')
LAB = ROOT / '01_labs/04_weather_forecasting'
SOURCE = LAB / 'source_code'
ASSETS = LAB / 'assets'
CACHE = Path(os.environ.get('AI4SCI_WEATHER_CACHE',
                            '~/.cache/ai4sci/weather')).expanduser().resolve()
DATA = CACHE / 'data-20220901'
COASTLINE = ASSETS / 'coastline.geojson'
COASTLINE_CREDIT = 'Natural Earth 1:110m coastline (public domain)'
CASE_TIME = '2022-09-01T00:00:00Z'
DEVICE = os.environ.get('AI4SCI_DEVICE', 'auto').strip().lower()
if DEVICE == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE not in {'cpu', 'cuda'}:
    raise ValueError('AI4SCI_DEVICE must be auto, cuda, or cpu.')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA was requested but is unavailable. Check the GPU and selected kernel.')
sys.path.insert(0, str(SOURCE))
RUN_COMPLETED = False
EVALUATION_COMPLETED = False
print('Python:', sys.executable)
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (slower execution)')
print('Cache:', CACHE)

def run_script(script, *arguments):
    command = [sys.executable, '-u', str(SOURCE / script), *map(str, arguments)]
    started = time.perf_counter()
    with subprocess.Popen(command, cwd=LAB, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    return time.perf_counter() - started

In [ ]:
preparation_seconds = run_script('prepare.py', '--cache-dir', CACHE)
provenance = json.loads((DATA / 'provenance.json').read_text())
assert provenance['initial_time_utc'] == CASE_TIME
print(f'Preparation and cache verification: {preparation_seconds:.1f} seconds')
print('The initial state and separate verification data are ready.')

입력은 이 체크포인트에서 제외하는 남극점 행을 빼고 원래의 0.25° 위경도 격자를 유지합니다. 상대습도는 문서화된 IFS 공식을 사용하여 ERA5 온도와 비습으로부터 계산합니다. `provenance.json`에 출처와 전처리를 기록합니다.

`initial.npz`에는 시작 상태만 들어 있습니다. `truth.npz`에는 검증에 사용하는 ERA5 장이 들어 있으며, 이 파일은 예측 스크립트에 전달하지 않습니다.

In [ ]:
with np.load(DATA / 'initial.npz', allow_pickle=False) as saved:
    initial_state = saved['state']
    print('Input shape (variable, latitude, longitude):', initial_state.shape)
    print('Initialization:', np.datetime_as_string(saved['time'], unit='h'), 'UTC')
    print('Variables:', ', '.join(saved['variables'].tolist()))
    print('Latitude:', saved['latitude'][0], 'to', saved['latitude'][-1], 'degrees')
    print('Longitude:', saved['longitude'][0], 'to', saved['longitude'][-1], 'degrees')
del initial_state

## 2. 48시간 예측

6시간 간격으로 여덟 단계를 실행합니다. **48시간은 예측 대상 기간이며, 실행을 기다리는 시간이 아닙니다.** 진행 메시지에 각 예측 선행 시간과 측정된 계산 시간이 표시됩니다.

실행할 때마다 별도의 결과 폴더에 저장합니다. 이 셀 전체의 시간 측정에는 모델 불러오기와 출력 저장이 포함되지만, 모델 계산만의 시간 측정에는 포함되지 않습니다.

In [ ]:
RUN_COMPLETED = False
EVALUATION_COMPLETED = False
RUN = LAB / 'outputs' / f'forecast-{uuid.uuid4().hex[:10]}'
forecast_seconds = run_script('run_forecast.py',
                              '--initial', DATA / 'initial.npz',
                              '--model-dir', CACHE / 'model',
                              '--output', RUN, '--device', DEVICE, '--steps', '8')
RUN_COMPLETED = True
print(f'Forecast, model loading and output saving: {forecast_seconds:.1f} seconds')

In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError('Complete the forecast cell before reading its results.')
runtime = json.loads((RUN / 'runtime.json').read_text())
print(f"Eight model steps only: {runtime['inference_seconds']:.2f} seconds")
if runtime['cuda_peak_allocated_gib'] is not None:
    print(f"Peak PyTorch-allocated GPU memory: {runtime['cuda_peak_allocated_gib']:.2f} GiB")
print('These measurements describe this run, not a guaranteed runtime on other hardware.')
with np.load(RUN / 'forecast.npz', allow_pickle=False) as saved:
    print('Saved variables:', ', '.join(saved['variables'].tolist()))
    print('Valid times (UTC):', np.datetime_as_string(saved['times'], unit='h').tolist())
print('Results:', RUN.relative_to(ROOT))

## 3. ERA5 및 지속성 예측과 비교

움직이는 기상 지도만으로 예측을 판단하기는 어렵습니다. 예측을 ERA5 및 **지속성 예측**과 비교하세요. 지속성 예측은 모든 미래 시각의 기상장을 초기 상태 그대로 유지합니다.

평가기는 전 지구와 **동아시아(북위 10–55°, 동경 100–160°)**에서 위도에 따른 면적 가중 RMSE를 보고합니다. 오차가 작을수록 좋습니다. 바람 벡터 오차는 두 수평 성분을 모두 사용하므로 풍속뿐 아니라 풍향의 차이도 측정합니다. 초기 프레임은 예측 점수에서 제외합니다.

기압 오차의 단위는 hPa, 온도 오차의 단위는 K입니다. 1 K의 차이는 1 °C의 차이와 같습니다. 이 사례의 점검 기준은 예측을 실행하기 전에 정했습니다. 이 기준은 순위표 점수나 현업 예측 정확성의 근거가 아닙니다.

In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError('Complete the forecast cell before evaluating it.')
EVALUATION_COMPLETED = False
EVALUATION = RUN / 'evaluation'
evaluation_seconds = run_script('evaluate_weather.py',
                                '--forecast', RUN / 'forecast.npz',
                                '--truth', DATA / 'truth.npz',
                                '--output-dir', EVALUATION,
                                '--coastline', COASTLINE,
                                '--coastline-attribution', COASTLINE_CREDIT,
                                '--acceptance', ASSETS / 'weather-acceptance.json', '--gif')
summary = json.loads((EVALUATION / 'evaluation.json').read_text())
EVALUATION_COMPLETED = True
print(f'Evaluation, maps and animation: {evaluation_seconds:.1f} seconds')
print('Reference:', summary['reference_kind'])
checks = summary['acceptance']['checks']
print(f"Prespecified case checks met: {sum(check['passed'] for check in checks)}/{len(checks)}")
print('Inspect the errors below, including locations where the forecast is less accurate.')

In [ ]:
if not EVALUATION_COMPLETED:
    raise RuntimeError('Complete the evaluation cell before viewing its results.')
for region, records in summary['metrics'].items():
    rows = [
        f"### {region.replace('_', ' ').title()}", '',
        'Each entry is **forecast RMSE / persistence RMSE**.', '',
        '| Lead | Wind vector (m/s) | Pressure (hPa) | Temperature (K) |',
        '|---:|---:|---:|---:|',
    ]
    for row in records:
        wind = row['wind_10m']
        pressure = row['variables']['msl']
        temperature = row['variables']['t2m']
        rows.append(
            f"| +{row['lead_hours']} h "
            f"| {wind['vector_rmse']:.2f} / {wind['persistence_vector_rmse']:.2f} "
            f"| {pressure['display_rmse']:.2f} / {pressure['display_persistence_rmse']:.2f} "
            f"| {temperature['rmse']:.2f} / {temperature['persistence_rmse']:.2f} |"
        )
    display(Markdown('\n'.join(rows)))
display(Image(filename=str(EVALUATION / 'rmse-by-lead-global.png')))
display(Image(filename=str(EVALUATION / 'rmse-by-lead-east_asia.png')))

## 4. 기상 지도 살펴보기

`variable`과 `lead_hours`를 바꾸고 다음 셀을 다시 실행하세요. **먼저 +6, +24, +48시간을 살펴보세요.** 표시 내용만 바뀌므로 예측을 다시 실행할 필요는 없습니다.

가장 강한 바람의 위치와 세기를 비교한 뒤 기압과 온도를 살펴보세요. 큰 규모의 패턴이 일치한다고 해서 모든 국지적 극값까지 맞는 것은 아닙니다.

예측과 재분석은 여덟 예측 선행 시간에 걸쳐 고정된 동일한 색상 척도를 사용합니다. 오차 패널은 별도의 대칭 척도를 사용합니다. 바람 화살표는 방향과 크기를 나타내고, 해안선은 지리적 위치를 파악하는 데 도움을 줍니다.

In [ ]:
import matplotlib.pyplot as plt
from evaluate_weather import (
    PLOT_UNITS, draw_map, geographic_lines, load_pair, regional_plot_data,
)

variable = 'wind_speed'  # 'wind_speed', 'msl', 't2m', 'u10m', or 'v10m'
lead_hours = 24          # 6, 12, 18, 24, 30, 36, 42, or 48

if not EVALUATION_COMPLETED:
    raise RuntimeError('Complete the evaluation cell before viewing its maps.')
if variable not in PLOT_UNITS or lead_hours not in range(6, 49, 6):
    raise ValueError('Choose a listed variable and a six-hour lead from 6 to 48.')
data = load_pair(RUN / 'forecast.npz', DATA / 'truth.npz')
plot = regional_plot_data(data, variable)
frame = data['lead_hours'].tolist().index(lead_hours)
lines = geographic_lines(COASTLINE)
field_names = {'wind_speed': '10 m wind speed', 'msl': 'Mean sea-level pressure',
               't2m': '2 m temperature', 'u10m': '10 m eastward wind',
               'v10m': '10 m northward wind'}

with plt.rc_context({'font.size': 12, 'axes.titlesize': 14}):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    for ax, key, title in zip(axes, ('reference', 'prediction', 'error'),
                              ('ERA5 reanalysis', 'FourCastNet forecast', 'Forecast minus ERA5')):
        is_error = key == 'error'
        limits = (-plot['error_limit'], plot['error_limit']) if is_error else plot['limits']
        vector = None
        if variable == 'wind_speed':
            vector = (plot['prediction_uv'][frame] - plot['reference_uv'][frame]
                      if is_error else plot[key + '_uv'][frame])
        image = draw_map(ax, plot[key][frame], plot['latitude'], plot['longitude'],
                         limits, 'RdBu_r' if is_error else plot['cmap'], lines, vector)
        ax.set_title(title)
        fig.colorbar(image, ax=ax, orientation='horizontal',
                     label=PLOT_UNITS[variable] + (' difference' if is_error else ''))
    valid_time = np.datetime_as_string(data['times'][frame], unit='h')
    fig.suptitle(f'{field_names[variable]} | +{lead_hours} h | {valid_time} UTC', fontsize=16)
    plt.show()
print(COASTLINE_CREDIT)
del data, plot

### 48시간 동안의 변화 보기

애니메이션은 **ERA5 재분석**, **FourCastNet 예측**, **예측 오차**를 나란히 보여줍니다. 움직이는 바람 패턴을 따라가며 위치와 세기의 차이를 찾아보세요. 표시 시각은 UTC입니다.

In [ ]:
if not EVALUATION_COMPLETED:
    raise RuntimeError('Complete the evaluation cell to create the animation.')
display(Image(filename=str(EVALUATION / 'east-asia-wind-6-to-48h.gif')))

## 결과 토의

1. +6, +24, +48시간을 비교하세요. 어떤 특징이 이동하거나 변하나요? 예측에서 그 특징이 ERA5와 같은 위치에 있나요?
2. FourCastNet은 바람, 기압, 온도에서 지속성 예측보다 나은가요? 지도뿐 아니라 오차 표도 사용하세요.
3. 영역 전체의 RMSE가 작아도 폭풍 중심의 큰 오차가 가려질 수 있을까요? 예를 들거나 무엇을 확인할지 설명하세요.
4. 모델이 자신의 출력을 반복해서 입력받을 때 오차가 커질 수 있는 이유는 무엇인가요? 모든 변수의 오차가 매 단계 증가해야 하나요?
5. 이 사전 학습 모델을 사용하는 것은 실습 1–3에서 PINN을 학습하는 것과 어떻게 다른가요?

**이 실험의 한계:** 하나의 과거 초기 상태를 사용하는 실험이며 실시간 예측 서비스가 아닙니다. 같은 초기 상태에서 얻은 여덟 예측 선행 시간은 독립적인 사례 여덟 개가 아닙니다. 재분석은 직접 관측이 아닙니다. 더 폭넓은 예측 성능을 검증하려면 더 많은 날짜와 기상 유형이 필요합니다.

이 기상 실습은 본 과정의 기존 실습 4를 대체합니다. [기존 Navier–Stokes PINN 노트북](../../ETC/reference_labs/04_navier_stokes/Lab_4_Navier_Stokes.ipynb)은 참고 자료로 보존되어 있으며, 검증된 기상 예측 자료가 아닙니다.

## 출처

- [NVIDIA FourCastNet 체크포인트](https://huggingface.co/nvidia/fourcastnet1): 사전 학습된 26채널 AFNO, Apache-2.0. 리비전 `c67a63995f6c8e0e557eb3d791f32f437e9b02d5`.
- [Earth2Studio FCN 구현](https://github.com/NVIDIA/earth2studio/blob/486c5daa98841b0ce93cb78cb70d0dee7a8a56e6/earth2studio/models/px/fcn.py): 변수 순서, 격자, 정규화.
- [ARCO-ERA5](https://github.com/google-research/arco-era5): ECMWF의 Copernicus Climate Change Service가 만든 ERA5를 Google 공개 아카이브를 통해 배포합니다. 로컬 `provenance.json`에 데이터 출처와 전처리를 기록합니다.
- [Natural Earth](https://www.naturalearthdata.com/about/terms-of-use/): 퍼블릭 도메인 해안선 자료.

지도는 기압을 hPa, 온도를 °C로 표시합니다. 보조 스크립트는 모델의 예측값을 보존하며, 기준 자료에 맞추어 예측을 조정하지 않습니다.

[시작 안내](../../Start_Here.ipynb) · 이전: [실습 3: 열전도](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · 다음: [도전 과제 1: 파동 역학](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)